# ST-OMR Meter V5-2T — Bounded Class-Balanced Head Repair

Single-run, fail-closed training harness pinned to exact CI-green implementation commit `8d98c1f6ad66ee896d28c02fb7ff1afafab23be9`. It runs the one preregistered 64-parameter head-weight solve and stops before Historical Retention, First-30, V5 VAL, and FINAL_HOLDOUT.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "8d98c1f6ad66ee896d28c02fb7ff1afafab23be9"
EXPECTED_CI_RUN_ID = 32669332005
V5_2R_HEAD = "85c0b0083792e8b9ec60ee632cfc7015e885d548"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = (
    MYDRIVE
    / "ST-OMR-D10"
    / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
)
for name, path in {
    "DATA_ROOT": DATA_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "M4A_ROOT": M4A_ROOT,
    "D10_ROOT": D10_ROOT,
}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(
    ["git", "-C", str(REPO), "remote"], text=True
).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(
    ["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"]
)
fetched_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
actual_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD mismatch: expected={EXPECTED_HEAD} actual={actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_2t_bounded_class_balanced_head_repair_v1 as repair
print("MODULE IMPORT = PASS")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
print("FROZEN CHECKPOINTS = PASS")
print("2-AI =", DIGIT2_FROZEN)
print("3-AI =", DIGIT3_FROZEN)

ANN_DIR = DATA_ROOT / "annotations"
V5_2R_REPORT = ANN_DIR / "v5_2r_train_class_margin_gradient_audit_v1.json"
V5_2R_ENVELOPE = ANN_DIR / f"v5_2r_execution_envelope_{V5_2R_HEAD}.json"
REPORT_PATH = ANN_DIR / repair.TRAINING_REPORT_NAME
CANDIDATE_DIR = ANN_DIR / repair.CANDIDATE_DIR_NAME
TEMP_CANDIDATE_DIR = ANN_DIR / repair.TEMP_CANDIDATE_DIR_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_2t_execution_envelope_{EXPECTED_HEAD}.json"
for path in (V5_2R_REPORT, V5_2R_ENVELOPE):
    if not path.is_file():
        raise RuntimeError(f"Required V5-2R evidence missing: {path}")
for path in (REPORT_PATH, CANDIDATE_DIR, TEMP_CANDIDATE_DIR, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("INPUT BINDING = PASS")
print("OUTPUT GUARD = PASS")

required_safety = {
    "single_fixed_training_entry": True,
    "automatic_second_configuration": False,
    "hyperparameter_sweep": False,
    "trainable_surface": "head.weight-only-64-parameters",
    "frozen_backbone": True,
    "frozen_head_bias": True,
    "runtime_threshold_tuning": False,
    "alternative_threshold_evaluated": False,
    "new_bbox": False,
    "new_crop_geometry": False,
    "new_spatial_heuristic": False,
    "reserve_v5_train_opened": False,
    "historical_validation_opened": False,
    "historical_retention_executed_by_this_module": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "production_promotion": False,
}
boundary = repair.safety_boundary()
for key, expected in required_safety.items():
    if boundary.get(key) != expected:
        raise RuntimeError(f"Safety boundary mismatch: {key}={boundary.get(key)!r}")
contract = repair.implementation_contract()
if contract.get("actual_data_execution_requires_exact_sha_colab_harness") is not True:
    raise RuntimeError("Exact-SHA harness requirement missing")
if contract["solver"].get("execution_authorized") is not True:
    raise RuntimeError("Single solver execution is not authorized")
print("SAFETY BOUNDARY = PASS")
print("SINGLE CONFIGURATION = PASS | HEAD.WEIGHT-ONLY = PASS")
print("HISTORICAL_RETENTION=CLOSED | FIRST-30=CLOSED | V5_VAL=CLOSED")
print("FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN")

started = time.time()
def progress(processed, total, phase):
    if processed == 1 or processed == total or processed % 2048 == 0:
        print(phase, f"{processed}/{total}", f"| elapsed={int(time.time() - started)}s")

report = repair.train_bounded_class_balanced_head_repair_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    v5_2r_report=V5_2R_REPORT,
    v5_2r_execution_envelope=V5_2R_ENVELOPE,
    confirmation=repair.APPROVAL_TOKEN,
    progress=progress,
)

if not REPORT_PATH.is_file():
    raise RuntimeError(f"Training report not written: {REPORT_PATH}")
report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("Saved report mismatch")
if report.get("numerical_integrity_gate", {}).get("gate") != "PASS":
    raise RuntimeError("Numerical integrity did not PASS")
for key, expected in required_safety.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Saved report safety mismatch: {key}={report.get(key)!r}")
candidate_sha256 = {}
for digit in ("2", "3"):
    item = report["per_specialist"][digit]
    fit = item["fit"]
    invariants = item["state_invariants"]
    geometry = fit["geometry_float32_copy_back"]
    termination = fit["lbfgs_termination"]
    if fit.get("trainable_parameter_count") != 64:
        raise RuntimeError(f"{digit}-AI trainable count mismatch")
    if invariants.get("changed_state_keys") != ["head.weight"]:
        raise RuntimeError(f"{digit}-AI changed-state mismatch")
    if not all(invariants.get(key) is True for key in ("only_head_weight_changed", "backbone_bit_identical", "head_bias_bit_identical")):
        raise RuntimeError(f"{digit}-AI frozen-state integrity failed")
    if fit.get("finite_non_increasing_objective") is not True:
        raise RuntimeError(f"{digit}-AI finite/non-increasing objective failed")
    if termination.get("final_gradient_finite") is not True:
        raise RuntimeError(f"{digit}-AI final gradient is non-finite")
    if geometry.get("gate") != "PASS" or geometry.get("head_angle_change_degrees", 999.0) > 15.0:
        raise RuntimeError(f"{digit}-AI geometry gate failed")
    candidate_path = Path(item["candidate"]["candidate_path"])
    if not candidate_path.is_file() or not item["candidate"].get("reload_verified"):
        raise RuntimeError(f"{digit}-AI candidate missing/unverified")
    actual_sha = v52b._sha_file(candidate_path)
    if actual_sha != item["candidate"].get("candidate_sha256"):
        raise RuntimeError(f"{digit}-AI candidate SHA mismatch")
    candidate_sha256[digit] = actual_sha

post_run_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(f"Post-run HEAD mismatch: {post_run_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository changed during run")
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
envelope = {
    "schema": "st-omr-meter-v5-2t-exact-sha-training-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head_before_run": actual_head,
    "actual_head_after_run": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_report_name": repair.TRAINING_REPORT_NAME,
    "training_report_sha256": report_sha256,
    "candidate_checkpoint_sha256": candidate_sha256,
    "numerical_integrity_gate": report["numerical_integrity_gate"],
    "per_specialist_geometry": {digit: report["per_specialist"][digit]["fit"]["geometry_float32_copy_back"] for digit in ("2", "3")},
    "per_specialist_lbfgs": {digit: report["per_specialist"][digit]["fit"]["lbfgs_termination"] for digit in ("2", "3")},
    "historical_preservation_claimed": False,
    "safety_boundary": {key: report[key] for key in required_safety},
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
envelope_sha256 = hashlib.sha256(ENVELOPE_PATH.read_bytes()).hexdigest()

print()
print("============================================")
print("V5-2T EXACT-SHA TRAINING RESULT")
print("============================================")
for digit in ("2", "3"):
    item = report["per_specialist"][digit]
    fit = item["fit"]
    print()
    print(f"========== {digit}-AI ==========")
    print("STATE INTEGRITY =", item["state_invariants"])
    print("INITIAL OBJECTIVE =", fit["initial_total_objective"])
    print("FINAL OBJECTIVE =", fit["float32_copy_back_total_objective"])
    print("GROUP BCE =", fit["final_group_mean_bce_float64"])
    print("GEOMETRY =", fit["geometry_float32_copy_back"])
    print("LBFGS EVIDENCE =", fit["lbfgs_termination"])
    print("V5 TRAIN METRICS =", item["v5_train_metrics_at_frozen_threshold"])
    print("HISTORICAL TRAIN METRICS =", item["historical_train_metrics_at_frozen_threshold"])
    print("CANDIDATE SHA256 =", candidate_sha256[digit])
print()
print("EXACT SHA EXECUTION = PASS")
print("HEAD =", post_run_head)
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", report_sha256)
print("EXECUTION ENVELOPE =", ENVELOPE_PATH)
print("ENVELOPE SHA256 =", envelope_sha256)
print("NUMERICAL INTEGRITY = PASS")
print("HISTORICAL PRESERVATION CLAIMED = False")
print("HISTORICAL RETENTION = NOT RUN")
print("FIRST-30 = CLOSED | V5 VAL = CLOSED | FINAL_HOLDOUT = LOCKED")
